# 14f · 把整条线讲出来

> 这一章是 [`docs/contributions.md`](../../docs/contributions.md) 的**中文大白话版**。
> 那份文档是写给看的人的（英文、一条一条摆证据）；这一章是写给**你自己**的——面试前一天读的那一遍。

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → [第二阶段 F：一条线，从一个分数到两个能量]
                 ├─ 14a 把稳定性变成能量 ✅
                 ├─ 14b 把测序读数变成两种测量 ✅
                 ├─ 14c 把一个分数拆成两个能量 ✅
                 ├─ 14d 把两个能量变成一张位点地图 ✅
                 ├─ 14e 把地图用到一个真实的药上 ✅
                 └─ 14f 把整条线讲出来  ← 你在这里

附录（不在这条线上）
                 └─ 14x 实验室该测多少个 ✅
```

## 这一章和别的章不一样

前面六章都在**产生数字**。这一章**一个新数字都不产生**，它只做三件事：

| | 做什么 |
|---|---|
| **1** | 把散在六章里的创新点，用大白话讲成一条线 |
| **2** | 把抽象的东西**现场算一遍**给你看——热力学读出层、识别性问题、能量相加、药救得回来，都是几行代码就能跑出来的小例子，跑过一遍你才真的懂，面试时才敢展开讲 |
| **3** | 按「药企面试官真正想听什么」重新排序，并说明**为什么这么排** |

所以这一章里引用的所有数字（0.368、2.8×、0.769……）都是从 14a–14e 抄过来的，**出处永远是那一章的 notebook 和 MLflow 记录**，不在这里重算。

---

## 0 · 先想清楚：面试官到底在问什么

你会遇到三类人，他们问的问题长得不一样，但底下是同一个担心。

| 面试官 | 他嘴上问 | 他心里想的 |
|---|---|---|
| **科学家 / 组长** | "这和 AlphaMissense 有什么区别？" | 这人是会跟着别人做，还是能自己发现问题 |
| **平台 / 工程负责人** | "你这个能上线吗？" | 交给他的东西，三个月后还跑不跑得动 |
| **老板 / 用人部门** | "这个能帮我们干什么？" | 花钱雇他，下个季度能多做成哪件事 |

三个担心合起来就一句话：

> **「这个人能不能在没人给答案的情况下，做出一个别人敢用的判断。」**

下面每一节都是在回答这一句。顺序不是按时间做的顺序，是按**他们最想听的顺序**。

---

## 准备

这一章只需要项目里的物理模块 `qafi_core.energy`，不碰数据库、不碰 AWS，在任何一台装了这个项目的机器上都能跑完。

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)

from qafi_core import energy

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 200)

print("RT =", energy.RT, "kcal/mol（30 摄氏度下的气体常数 x 温度，酵母筛选的温度）")

---

## 1 · 创新点一：输出的是**两个能量**，不是一个分数

### 大白话

一辆车开不动了，修车师傅会告诉你「发动机坏了」还是「轮胎没气了」。

现在整个领域的标准输出是**一个 0 到 1 的分数**，相当于只告诉你「这辆车不太行，打 0.87 分」。分数高低能帮你**排序**——先修哪辆——但它**不告诉你修哪儿**。

蛋白质的"开不动"只有两种：

| 哪儿坏了 | 物理量 | 能不能救 |
|---|---|---|
| **折不起来**（蛋白散架了，细胞当垃圾清掉） | ΔΔG_fold，折叠自由能 | 可能可以：小分子伴侣抱住它、矫正剂帮它折 |
| **折得起来，但不干活了**（活性位点被改了） | ΔΔG_func，功能自由能 | 基本没救，得换别的路子 |

我们的模型**直接输出这两个数**，单位是 kcal/mol——物理学家和做实验的人都认的单位，不是我们自己编的分数。

### 为什么这不是"多加一个输出头"那么简单

你可能会想：那我在网络最后接两个输出不就行了？

不行。**网络没有任何理由把两个输出分别对应到折叠和功能上**——它可以把能量全放第一个头里，第二个头输出 0，损失函数一样低。要让分离真的发生，得靠别的东西来**强制**它。

我们靠的是**实验本身的物理**：

### 物理在哪里：读出层（readout layer）

关键的一句话：**实验从来测不到能量，它只能测到一个表型。而测什么表型，决定了你能看见哪个能量。**

```
丰度实验（VAMP-seq、Domainome、AbundancePCA）
    细胞里还剩多少这个蛋白  →  只看得见「折叠好的那一份」      →  只含 ΔΔG_fold

活性 / 结合实验
    这个蛋白还能不能干活    →  看得见「折叠好 而且 功能完好的那一份」→  两个能量都含
```

所以：

1. 丰度实验把 **ΔΔG_fold 钉死了**；
2. 活性实验里**超出丰度能解释的那部分**，就只能是 ΔΔG_func。

功能能量不含稳定性，**是构造上保证的，不是网络被"要求"做到的**。

从能量到"看得见的份额"，走的是玻尔兹曼分布——高中化学那个 $e^{-\Delta G/RT}$。下面把它画出来。

In [ ]:
g = np.linspace(-4, 4, 400)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

# 左：丰度实验看到的东西
axes[0].plot(g, energy.fraction_folded(g), color="#2a7ab9", lw=2)
axes[0].axvline(2.0, color="grey", ls="--", lw=0.9)
axes[0].annotate("典型野生型\ng_fold ≈ 2 → 折叠 97%", xy=(2.0, energy.fraction_folded(2.0)),
                 xytext=(-3.8, 0.80), fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="grey", lw=0.8))
axes[0].set(xlabel="g_fold（kcal/mol，越大越稳）", ylabel="折叠好的分子占比",
            title="丰度实验看到的：只有折叠", ylim=(-0.03, 1.03))

# 右：活性实验看到的东西
for g_func, color in ((2.0, "#1a9850"), (0.5, "#e08b3a"), (-1.0, "#d73027")):
    axes[1].plot(g, energy.fraction_functional(g, g_func), color=color, lw=2,
                 label=f"g_func = {g_func:+.1f}")
axes[1].plot(g, energy.fraction_folded(g), color="#2a7ab9", lw=1.2, ls=":", label="（折叠占比，对照）")
axes[1].set(xlabel="g_fold（kcal/mol）", ylabel="又折叠又能干活的分子占比",
            title="活性实验看到的：折叠 AND 功能", ylim=(-0.03, 1.03))
axes[1].legend(fontsize=8.5, loc="upper left")

for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### 这两张图怎么读

**左图只有一条线。** 丰度实验的读数完全由 `g_fold` 决定——功能好不好，它根本看不见。所以丰度数据是"干净的折叠能样本"。

**右图有三条线。** 同样的折叠能，功能能不同，活性读数就完全不同。两件事值得注意：

- **红线（功能坏了，g_func = −1.0）封顶在 0.16。** 蛋白再稳也没用——折叠做到 100% 也只有 16% 的分子能干活。这就是"折得好 ≠ 有用"。
- **绿线（功能很强，g_func = +2.0）在左半边反而跑到了折叠占比（蓝虚线）上面。** 这不是画错了，这是三态耦合平衡的真实物理：一个非常有利的结合会**把不稳定的蛋白拉进折叠态**（勒夏特列原理）。我们用的是三态模型而不是"折叠占比 × 功能占比"的简单相乘，就是为了把这件事表示对。

两张图合起来就是那句关键的话：**丰度实验给你一个只含折叠的方程，活性实验给你一个含两者的方程；两个方程，两个未知数。**

> **面试时可以说的一句**："分离不是我在架构上建议的，是测定类型强制的——丰度实验在数学上就只含一个能量。"

### 这件事是谁先做的，我们新在哪

这套热力学模型不是我们发明的，是 **Lehner 组的 ddPCA 和 MoCHI**。必须讲清楚，不然就是抢功劳。

**我们新的是方向**：

| | MoCHI / ddPCA | 我们 |
|---|---|---|
| 做什么 | 把能量**拟合**到某一个蛋白的实测数据上 | 从序列和结构**预测**能量 |
| 需要什么 | 这个蛋白的**双突变**实验数据 | 只要序列和结构 |
| 能回答没数据的蛋白吗 | **不能** | **能** |

一句话：**他们是测量工具，我们是预测工具。**

---

## 2 · 证据：在一个模型从没见过的蛋白上，两个数真的分开了

光说"我们输出两个数"没用，得证明这两个数**各自对应各自的东西**。这件事只有在一个地方能真测——需要有人在**同一个蛋白**上，既测了折叠自由能，又测了结合自由能，还都换算成了 kcal/mol。

**KRAS 就是这个地方。** Weng 等 2024 从 26,000 次测量里推出了每个变体的 1 个折叠自由能 + 对 6 个伙伴蛋白的结合自由能，一共 **7 个实测能量**。

而且 KRAS 对我们是干净的：KRAS 和它唯一的近亲 HRAS **都在我们锁死的考试集那一边**，训练时一个都没见过。

In [ ]:
kras = pd.DataFrame(
    [
        ("我们的折叠能",   0.368, 0.202),
        ("我们的功能能",   0.245, 0.310),
        ("AlphaMissense",  0.293, 0.382),
        ("CPT-1",          0.336, 0.340),
        ("ESM-2 对数几率", 0.366, 0.362),
    ],
    columns=["预测值", "对实测折叠", "对实测结合（6 个的中位数）"],
)
kras["偏向（折叠 − 结合）"] = (kras["对实测折叠"] - kras["对实测结合（6 个的中位数）"]).round(3)
display(kras.style.hide(axis="index").format(precision=3))

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.4))
colors = ["#2a7ab9", "#d73027", "#999999", "#999999", "#666666"]
for (_, r), color in zip(kras.iterrows(), colors):
    ours = "我们" in r["预测值"]
    ax.plot([0, 1], [r["对实测折叠"], r["对实测结合（6 个的中位数）"]],
            "-o", color=color, lw=2.6 if ours else 1.4, ms=7 if ours else 5,
            alpha=1.0 if ours else 0.75, zorder=3 if ours else 2)
    ax.annotate(r["预测值"], xy=(1.02, r["对实测结合（6 个的中位数）"]), va="center",
                fontsize=9.5, color=color, fontweight="bold" if ours else "normal")

ax.set_xticks([0, 1])
ax.set_xticklabels(["对实测\n折叠自由能", "对实测\n结合自由能"], fontsize=10)
ax.set_xlim(-0.12, 1.55)
ax.set_ylabel("Spearman 相关（符号已对齐，越高越好）")
ax.set_title("同一份 KRAS 数据，问两个问题：线斜向哪边，就是它偏向哪个机制", fontsize=11)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

### 这张图怎么读——这是整个项目最该练熟的一张

**看线的斜率，不要看线的高低。**

- **灰色的三条基本是平的**。ESM-2 偏向只有 +0.004，CPT-1 只有 −0.004。意思是：你问它"这个突变是把蛋白折散了还是把功能弄坏了"，它**给你同一个答案**。它做不到不是因为它弱——它是个很强的模型——而是因为**一个数只有一个值，只能有一个偏好**。
- **我们的两条线斜向相反**，中间差 0.23。折叠能偏向折叠（+0.167），功能能偏向结合（−0.066）。
- 7 个实测能量，**每一个都是对应的那个能量赢**；按位置重抽（同一位置的突变并不独立），**7 个里 6 个显著**。

### 必须主动说的一句

**AlphaMissense 在结合上比我们的功能能准**（0.382 对 0.310）。

主动说出来，不是谦虚，是因为：

1. 面试官大概率知道，你不说他会自己查出来，那时你的所有数字都打折；
2. 这正好把话题引到真正的点上——**我们声称的不是更准，是输出可以被"问机制"**。准确度和可问性是两件事。

> **一句话版**："我们没有在任何一个单项上赢。我们赢的是：我们的输出能回答一个他们的输出在数学上回答不了的问题。"

---

## 3 · 创新点二：我们找到了这件事**通常为什么会失败**，并解决了它

**这一节是整个项目里最值钱的一段**，因为它是唯一能证明「这人能自己做研究」的证据。前面的东西说到底是"实现了一个已知的想法"，这一段是"遇到了一个没人给答案的问题"。

### 大白话

两个人合伙做了一锅汤，你只能尝最后那一口。汤咸了——**是甲放多了盐，还是乙放多了盐？**

你尝不出来。因为你只尝到了**结果**，而同一个咸度，有无数种"甲放多少、乙放多少"的组合可以凑出来。

活性实验就是这锅汤：它只看见"折叠 × 功能"的**乘积**。读数很低，可能是"压根没折起来"，也可能是"折得好好的但不干活"。

这在统计上有个名字：**不可识别（not identifiable）**。下面画出来。

In [ ]:
TARGET = 0.30  # 活性实验读到的数值

def g_func_for(g_fold, target=TARGET, rt=energy.RT):
    """给定折叠能和一个目标读数，反解出对应的功能能——这就是「同一个读数的无数种解释」。"""
    return -rt * np.log((1 / target - 1) / (1 + np.exp(-g_fold / rt)))

gf = np.linspace(-2.0, 5.0, 300)
gu = np.linspace(-2.0, 2.5, 300)
GF, GU = np.meshgrid(gf, gu)
Z = energy.fraction_functional(GF, GU)

fig, ax = plt.subplots(figsize=(7.2, 5.2))
shade = ax.contourf(GF, GU, Z, levels=12, cmap="Blues", alpha=0.75)
fig.colorbar(shade, ax=ax, label="活性实验读到的数值")
line = ax.contour(GF, GU, Z, levels=[TARGET], colors="#d73027", linewidths=2.6)
ax.clabel(line, fmt={TARGET: f"读数都 = {TARGET:.2f}"}, fontsize=9.5, manual=[(-0.3, 0.05)])

for g_fold, tag in ((-1.0, "A  折不起来\n但功能很强"),
                    (1.0, "B  两边都\n有点问题"),
                    (4.0, "C  折得完美\n但功能坏了")):
    g_func = g_func_for(g_fold)
    ax.plot(g_fold, g_func, "o", color="black", ms=8, zorder=5)
    ax.annotate(tag, xy=(g_fold, g_func), xytext=(g_fold - 0.55, g_func + 0.42), fontsize=9, zorder=6,
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="grey", alpha=0.92))
    print(f"{tag[:1]}:  g_fold={g_fold:+.2f}  g_func={g_func:+.2f}  →  "
          f"折叠好的分子占 {100 * energy.fraction_folded(g_fold):5.1f}%  ，"
          f"活性读数 {energy.fraction_functional(g_fold, g_func):.3f}")

ax.set(xlabel="g_fold（折叠，kcal/mol）", ylabel="g_func（功能，kcal/mol）",
       title="红线上每一点的实验读数完全相同——\n只给活性数据，模型没有任何理由选 A 而不选 C")
fig.tight_layout()
plt.show()

### 读一下刚才打印的三行

A 的蛋白**只有 16% 折得起来**，C 的蛋白**99.9% 都折得好好的**——两者差了整整 5 kcal/mol。可是**活性实验给出的读数一模一样，都是 0.30**。

它们是**完全相反的两种病**：

| | A | C |
|---|---|---|
| 病在哪 | 蛋白散架了，活性位点完好 | 蛋白好好的，活性位点废了 |
| 药能不能救 | **有可能**（伴侣分子、矫正剂） | **基本不能** |
| 活性实验看到的 | **0.30** | **0.30** |

只给活性数据，拟合出 A 和拟合出 C，损失函数是**同样低的**。这就是为什么"多加一个输出头"注定失败。

实测下来，两个预测能量的相关系数是 **0.82**——等于**根本没分开**。

### 两个标准解法，都试了，都不行，都留在了记录里

| 试的办法 | 想法 | 结果 |
|---|---|---|
| 给功能能加 **L1 惩罚**（稀疏化） | 既然分不开，那就逼功能能"没事别乱动" | **它直接塌成 0**（Spearman 0.001）。折叠能把所有东西吸收掉了，总分一点没变——等于退回了一个分数 |
| **钉住折叠能**，不让活性数据动它（MoCHI 的两步拟合，写成一个网络） | 先用丰度数据定死折叠，再让活性数据只管功能 | 相关系数 0.82 → **0.79**。几乎没动 |

**负面结果我们留着没删**，这本身就是一个可讲的点（见第 7 节）。

### 真正的解法：只有"两头都测过"的蛋白才有资格教模型分离

回到做汤：你尝不出谁放的盐——**除非你见过甲一个人做的那锅汤**。

有了甲单独做的那锅（= 丰度实验，只含折叠），你就知道甲的咸度；合起来那锅多出来的咸度，就只能是乙的。

所以关键问题变成：**训练集里有几个蛋白，是用丰度和活性两种方式都测过的？**

答案是 **9 个**。

我们做的事就是：**整个模型在全部数据上训练，但那一层把能量翻译成读数的"读出层"，只在这 9 个蛋白上拟合。**

| | 结果 |
|---|---|
| 代价 | 单项折叠精度从 0.430 掉到 0.368 |
| 收益 | 两个能量**真的分开了**（就是上一节 KRAS 那张图） |

### 为什么这一段面试官最爱听

因为它完整地走了一遍真正的研究流程：

```
做了显然的版本  →  发现它不工作（0.82 相关）
                →  想清楚为什么不工作（不可识别，不是调参问题）
                →  试两个标准解法，用数字证明都没用
                →  找到问题的真正约束（成对测量）
                →  付出代价换取目标，并把代价说出来
```

**而且这同时解释了 14b 为什么存在。** 成对的"丰度 + 结合"数据正好是 ddPCA 实验产出的东西，多拿几个这样的蛋白，是这个模型最能被改进的一点——所以我们自己搭了一条从原始测序数据做到适应度的流水线。创新点之间是连着的，不是堆出来的。

---

## 4 · 创新点三：能量可以相加，分数不行

### 大白话

分数是**打分**，能量是**账**。

打分不能相加：两个演员各打 8 分，一起演不等于 16 分。
但账可以相加：你欠 A 一百，欠 B 一百，你就是欠两百。**这是热力学，不是经验规律。**

蛋白工程项目几乎永远在**叠突变**——改五个位置、改十个位置。所以"能不能把单突变的结果加起来预测组合"直接决定这个模型在项目里有没有用。

下面现场算一个例子，这个例子比任何表格都说明问题。

In [ ]:
# 一个很稳的野生型
G_FOLD_WT = 4.0
share_wt = energy.fraction_folded(G_FOLD_WT)

# 两个突变，每个让折叠能掉 2.5 kcal/mol
ddg1 = ddg2 = -2.5

single = energy.fraction_folded(G_FOLD_WT + ddg1) / share_wt          # 单突变的相对丰度
truth  = energy.fraction_folded(G_FOLD_WT + ddg1 + ddg2) / share_wt   # 双突变，能量相加
naive  = single * single                                              # 把「表型」相乘（= 适应度相加）

print(f"野生型：g_fold = {G_FOLD_WT:+.1f} → 折叠占比 {share_wt:.4f}")
print()
print(f"单突变 1（ddG = {ddg1:+.1f}）：相对丰度 {single:.3f}   ← 看起来几乎没事，只掉了 {100 * (1 - single):.1f}%")
print(f"单突变 2（ddG = {ddg2:+.1f}）：相对丰度 {single:.3f}   ← 同上")
print()
print(f"双突变，把能量相加（真实物理）：{truth:.3f}   ← 蛋白基本没了")
print(f"双突变，把表型相乘（常见做法）：{naive:.3f}   ← 预测它几乎没事")
print()
print(f"错了 {naive / truth:.1f} 倍。单看两个单突变，你会以为这两个改动都很安全。")

### 这就叫**饱和**

野生型太稳了（折叠占比 99.9%），掉 2.5 kcal/mol 还剩 92%——**实验上几乎看不出区别**。于是两个单突变看起来都"安全"。

但能量账是实打实的：掉 2.5 再掉 2.5 就是掉 5，蛋白直接翻过了折叠的悬崖，只剩 16%。

**表型会饱和，能量不会。** 把饱和了的表型相加，等于在用一把已经顶到头的尺子量东西。

### 真数据上的结果

用 PSD95-PDZ3 的 ddPCA 数据：只用**单突变**定出每个替换的两个能量，然后预测大约 5,000 个**从没拟合过**的双突变。

| 做法 | 丰度 R² | 结合 R² |
|---|---|---|
| **能量相加** | **0.742** | **0.840** |
| 适应度相加 | −0.041 | −0.127 |
| 两个单突变里更糟的那个 | 0.675 | 0.711 |

**R² 是负的**，意思是：还不如直接猜所有双突变都等于平均值。原因就是上面那个例子。

> **一句话版**："自由能是可加的，这是热力学给的，不是我拟合出来的。所以单突变数据就能外推到组合——这是蛋白工程项目唯一真正需要的外推。"

---

## 5 · 创新点四：一个真药上的验证，而且方向被数据纠正了

### 大白话

前面都是"我们的输出能说机制"。那**机制到底有没有用**？

有一类药正好只对一种机制有效，它叫**药理伴侣**（pharmacological chaperone）。它不抑制蛋白，它在蛋白折叠的时候**抱住它**，让它别散架、别被细胞当垃圾清掉。

所以按理说：它只救"**功能还在、只是折不好**"的突变。

**法布里病**的药 migalastat（Galafold）2016–2018 年获批。获批前药厂在细胞里一个一个测了它对哪些 GLA 突变有效，并且**把有效和无效的名单都公开了**——这在药物数据里非常罕见（大部分只公布有效的那一半）。而 GLA 对我们完全干净：不在任何切分里，和训练集里的蛋白也不同源。

先用物理算一下"伴侣药应该救谁"。

In [ ]:
RESCUE = 1.5  # 假设伴侣药给折叠能加 1.5 kcal/mol

cases = [
    ("A  轻度失稳，功能完好", -0.8, -0.1),
    ("B  重度失稳，功能完好", -3.5, -0.1),
    ("C  折叠正常，功能坏了", -0.1, -2.5),
]
rows = []
for name, d_fold, d_func in cases:
    before = energy.fraction_functional(2.0 + d_fold, 1.0 + d_func)
    after = energy.fraction_functional(2.0 + d_fold + RESCUE, 1.0 + d_func)
    rows.append({"变体": name, "ΔΔG_fold": d_fold, "ΔΔG_func": d_func,
                 "用药前活性": round(before, 3), "用药后活性": round(after, 3),
                 "救回了": f"+{100 * (after - before):.1f} 个百分点"})
display(pd.DataFrame(rows).style.hide(axis="index"))

物理给出的预期：**A 救得最多，C 基本白搭**（折叠本来就好，再加稳定性没有意义）。

**B 很有意思**——折叠坏得太狠，加 1.5 kcal/mol 还是不够把它拉回折叠的悬崖这一侧，所以也救不回多少。

### 真数据上的结果

555 个带标签的 GLA 替换（254 个可救回 / 301 个不可救回），按位置分折做交叉验证：

| 用什么打分 | AUROC |
|---|---|
| **我们的功能能，单独用** | **0.759** |
| ESM-2 零样本 | 0.729 |
| 埋藏度 × 保守性 | 0.715 |
| 我们的折叠能，单独用 | 0.693 |

| 组合 | AUROC |
|---|---|
| **两个能量 + 埋藏度** | **0.769** |
| 两个能量 | 0.764 |
| ESM-2 | 0.725 |

### 最该讲的是这里：我们原本猜错了方向

我们一开始的假设是"**折叠坏了、功能好的**最容易被救"。数据说的是另一回事：

| | 可救回（254 个） | 不可救回（301 个） |
|---|---|---|
| 预测折叠能 | **−0.81** | −1.35 |
| 预测功能能 | **−0.14** | −0.58 |

能被救回的是「**只是轻微失稳 + 活性位点完好**」，不是「折叠全坏但功能完好」。

**为什么**：migalastat 本身就结合在活性位点上——它是用底物类似物当伴侣的。所以活性位点一坏，药连抓都抓不住；而折叠要是坏得太狠，一点点稳定化也扶不起来（就是上面算的 B）。

> **面试时主动讲这个。** 一个"我有假设 → 数据否了我 → 我找到了物理原因"的故事，比十个对的预测更能说明你会做研究。

---

## 6 · 创新点五：从"每个突变一个数"变成"每个位置是干什么的"

### 大白话

前面都在讲**变体**。但项目里的人真正问的问题往往是关于**位置**的：

> "我这个靶点，哪些残基是骨架不能动，哪些是干活的机器，哪些随便改？"

把一个位置上 19 种替换的两个能量各自平均一下，这个位置就有了两个数，于是能分成四类：

| 这个位置 | 意思 | 项目里怎么用 |
|---|---|---|
| **结构关键** | 一改就折不起来 | 别碰。想提稳定性反而要看这里 |
| **功能关键** | 折得起来，但一改就不干活 | 这是机器所在。**抑制剂和变构口袋的候选** |
| **两者都是** | 两个能量都很贵 | 最核心的位置 |
| **可改** | 都不贵 | 改造、加标签、做人源化的自由区 |

**注意：一个分数做不出这个分类。** 一个分数只能告诉你"这个位置重不重要"，所有重要的位置都长一个样。

### 验证一：拿 UniProt 的注释去比（模型从没见过这些注释）

118 个蛋白、103,744 个位置、197 万个替换。

In [ ]:
enr = pd.DataFrame(
    [("两者都是", 17_820, 9.19), ("功能关键", 8_164, 2.84),
     ("结构关键", 8_164, 0.21), ("可改", 69_596, 0.08)],
    columns=["位置类别", "位置数", "富集倍数（odds ratio）"],
)

fig, ax = plt.subplots(figsize=(7.6, 3.6))
colors = ["#6a3d9a", "#d73027", "#2a7ab9", "#aaaaaa"]
bars = ax.barh(enr["位置类别"], enr["富集倍数（odds ratio）"], color=colors, height=0.6)
ax.axvline(1.0, color="black", lw=1.1, ls="--")
ax.text(1.12, 3.40, "1 倍 = 和随机一样", fontsize=8.5, color="black")
ax.set_xscale("log")
ax.set_xlabel("在 UniProt 标注的「活性位点 / 结合位点」上的富集倍数（对数轴）")
ax.set_title("两个能量把蛋白组切开了——一个分数做不到这件事", fontsize=11, pad=12)
ax.set_xlim(0.05, 40)
for bar, v, n in zip(bars, enr["富集倍数（odds ratio）"], enr["位置数"]):
    ax.text(v * 1.15, bar.get_y() + bar.get_height() / 2,
            f"{v:.2f}×   （{n:,} 个位置）", va="center", ha="left", fontsize=9)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

**这张图的重点是"分开了"，不是"富集了"。**

"功能关键"富集 2.84 倍、"两者都是"富集 9.19 倍，而"结构关键"**反过来被耗竭到 0.21 倍**——也就是说，结构关键的位置**显著地不是**活性位点。

> **一个分数不可能产生这张图**。因为一个分数只有一个方向，它会把所有"重要"的类别**往同一边推**。要出现"一类富集、另一类耗竭"，你必须有两个互相独立的轴。

### 验证二：已经被药物占住的口袋，模型事先点不点得亮

这是一次**前瞻性**检验——模型没见过任何药、任何口袋。看这些口袋的残基，在**它自己那个蛋白内部**按功能代价排第几（0 = 最敏感）。

In [ ]:
pockets = pd.DataFrame(
    [("KRAS switch-II（索托拉西布）", 17, 0.238, 0.532, 0.0011, "✅ 落在自己蛋白的前四分之一"),
     ("ABL1 肉豆蔻酸口袋（阿西米尼）", 18, 0.232, 0.506, 0.0017, "✅ 落在自己蛋白的前四分之一"),
     ("SHP2 隧道（SHP099）", 15, 0.504, 0.498, 0.3900, "❌ 和随机没区别")],
    columns=["口袋", "残基数", "口袋内排名", "蛋白其余部分", "p", "结果"],
)
display(pockets.style.hide(axis="index").format({"口袋内排名": "{:.3f}", "蛋白其余部分": "{:.3f}", "p": "{:.4f}"}))

三个里两个点亮了，而且 p ≈ 0.001。

### 第三个没点亮——这一条要**主动讲**

SHP2 的变构隧道**没被找出来**（p = 0.39）。原因很具体，不是"模型不够好"：

> SHP2 的那个口袋是一个**构象开关**——它卡在三个结构域的交界处，作用是把蛋白维持在自抑制的构象里，**而不是在折叠好的状态下做化学反应**。我们的模型表示的是"折叠态"和"功能态"两个状态，构象开关不在这个表示里。

**知道自己方法的边界在哪，本身就是方法的一部分。** 面试官问"这个方法什么时候不work"，你有一个具体、有物理原因的答案，比说"效果还行"强十倍。

### 而且这个地图是**上线了的**，不是一张图

103,744 行进了 S3 的湖，有 Glue 表和 `sites_current` 视图，API 上有 `/v1/proteins/{protein}/sites` 这个路由，agent 上有一个 `get_site_map` 工具。

一个顺带的检查：KRAS 的三个致癌热点 G12、G13、Q61，在它自己 189 个位置里按功能代价排第 **11、26、44** 位。

---

## 7 · 创新点六：回答"我到底要测多少个才值"——用一个数字

### 大白话

这一章和模型本身关系最小，但**用人部门最爱听**，因为它直接是钱。

一个新项目拿到一个靶点，老板问："做一轮深度突变扫描要花几十万，到底值不值？"

过去的回答是"看情况"。我们把它变成一个数：**92 个考试集蛋白、128 个实验，当成 128 个"新项目"重放一遍**。

In [ ]:
lown = pd.DataFrame(
    {"测了 24 个": [0.420, 0.175, 0.323, 0.347],
     "测了 48 个": [0.420, 0.272, 0.388, 0.414],
     "测了 96 个": [0.419, 0.396, 0.448, 0.479],
     "测了 384 个": [0.421, 0.579, 0.558, 0.595]},
    index=["零样本（一个都不测）", "独热编码 + 岭回归", "ESM-2 + 岭回归", "ESM-2 + 零样本 + 岭回归"],
)

budgets = [24, 48, 96, 384]
fig, ax = plt.subplots(figsize=(7.8, 4.4))
styles = {"零样本（一个都不测）": ("#000000", "--", 2.2),
          "独热编码 + 岭回归": ("#aaaaaa", "-", 1.6),
          "ESM-2 + 岭回归": ("#2a7ab9", "-", 1.8),
          "ESM-2 + 零样本 + 岭回归": ("#d73027", "-", 2.4)}
for name, row in lown.iterrows():
    color, ls, lw = styles[name]
    ax.plot(budgets, row.values, ls, color=color, lw=lw, marker="o", ms=5, label=name)

ax.axvline(96, color="#1a9850", lw=1.2, ls=":")
ax.annotate("一块 96 孔板\n= 开始划算的点", xy=(103, 0.22), fontsize=9, color="#1a9850")
ax.set_xscale("log")
ax.set_xticks(budgets)
ax.set_xticklabels([str(b) for b in budgets])
ax.minorticks_off()
ax.set(xlabel="先测了多少个变体", ylabel="对剩下变体的排序能力（Spearman 中位数）",
       title="测多少个，模型才比「一个都不测」强", ylim=(0.12, 0.66))
ax.legend(fontsize=9, loc="upper left")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

三条可以直接说出口的结论：

| 结论 | 为什么对项目有用 |
|---|---|
| **少于约 50 个，别测。** 免费的零样本分数比任何在这些数据上拟合出来的模型都强 | 省钱，而且是个反直觉的建议——说明你是真算过的 |
| **一块 96 孔板是分水岭**，从这儿开始测量才开始回本 | 这是一个实验员听得懂的单位 |
| **独热编码要多花大约 4 倍的量**才追得上 | 说明"用语言模型表示"这件事值多少钱 |

### 接下来那个问题：那**该测哪 96 个**

模拟一次真实的设计循环：测一批 → 重新拟合 → 选下一批 → 再测。每轮 96 个，做 4 轮。

| 选法 | 找到的最好变体（占全局最好的比例） | 找到了几个全局前 1% |
|---|---|---|
| 随机挑 | 0.960 | 4 |
| 贪心（挑预测最好的） | 0.998 | 6 |
| **UCB（兼顾模型没把握的）** | **0.999** | **7** |

**同样的预算，多找到 75% 的顶尖变体。** 这是主动学习在蛋白工程里最朴素的用法，也是最容易让面试官点头的一条。

---

## 8 · 工程上的创新：这部分决定"能不能用"，而且很多候选人没有

科学部分决定能不能通过科学家那一关，**这一节决定能不能通过平台负责人那一关**。而且这一节很难装——要么你做过，要么你没做过。

### 8.1 验证的严谨程度（最该先讲的一条）

这个领域最容易出错、也最容易作弊的地方就是**考试集**。所以把它说明白：

| 我们做的 | 为什么 |
|---|---|
| **整个蛋白留出，连同它的亲戚** | 按变体切分一定漏题：同一个蛋白两边都有，那是背答案不是预测。相似序列用局部比对分组，**整组**去一边 |
| **考卷选一次、写进 dbt seed、只开一次**（2026-10-14，MLflow 有记录） | 一份被看过两次的考卷就不是考卷了 |
| **每一章都要拿自己带来的新数据再查一遍** | 14a 查出 **56 个** mega-scale 结构域和锁死的考试蛋白同源，于是**又训练了一个去污染的模型**给后面所有章用，并且**把代价写出来**（0.010 Spearman） |
| **和别人比较时，用别人自己的切分** | 14a 直接用 ThermoMPNN 发表的 `mega_splits.pkl`，所以我们的数字可以**直接摆在**他们、SPURS、StaB-ddG 旁边。自己发明一个切分，这三个比较就全废了 |
| **用之前先对账** | 14a 把发表的过滤链一行一行复现（776,298 → 607,839 → 391,090 → 272,712），并把论文里差的 140 行追到了野生型对照行上。14b 逐个变体对账，Pearson **1.0000** |
| **负面结果留着** | L1 惩罚、钉住折叠能、08b 的图网络、08c 的 LoRA、SHP2 那个找不到的口袋——全都在文档里 |

> **第三条是最值钱的。** "我查出来自己的新数据污染了考试集，所以我重训了一个干净的，代价是 0.010" ——大多数论文根本不会去查。这句话一说，面试官对你后面所有数字的信任度就不一样了。

### 8.2 整条链是通的

```
原始测序 reads  →（Nextflow + DiMSum，跑在 AWS Batch 上）→ 适应度
                →（两个能量的模型）→ 2.18 亿个变体全打分
                → S3 数据湖 → Athena → API → agent
```

| | 这里具体指什么 |
|---|---|
| **一切都有版本** | 输入按 release 号钉死，由 Lambda 从发布方拉进 S3 并写回执；模型进 Unity Catalog 注册表、过考试集闸门；这几份文档里引的每个数字都有 MLflow 记录 |
| **算力可搬** | reads→fitness 那条流水线，笔记本上用 Docker 跑、AWS Batch 上跑，**只换一个 profile**，不改代码；容器钉在原作者用的版本上 |
| **训练和服务用同一套特征** | 一个 SQL 宏同时给训练和线上算残基特征，外加一个一致性测试——训练/服务不一致是部署 ML 最常见的隐形故障 |
| **测过** | 263 个单元测试，包含把论文数字钉死的黄金测试，以及 CI 里一次不带数据的 Nextflow stub 跑通 |
| **运维过** | 88 个 Terraform 资源、用 OIDC 不存密钥、CloudWatch 告警、一个月几美元、按请求付费 |

### 8.3 一个很好用的小故事

公共镜像仓库限流，**6 次推送里有 3 次把 CI 搞挂了**。

很多人的做法是加重试。我们的做法是**结构性修复**：把镜像缓存起来、以版本号为 key，Dockerfile 把它作为构建参数接进来；然后**验证两条构建路径产出的镜像摘要完全相同**（`sha256:fad676…`）。

这个故事短、具体，而且刚好展示了"修根因 + 验证修复"这两件事。

---

## 9 · 我们**不**声称什么

这一节要和前面的声称说得一样清楚，因为面试官一定会找。而且**你先说出来，这些话的杀伤力就没了**。

| 不声称 | 具体数字 |
|---|---|
| **不是最准的稳定性预测器** | 每结构域中位 Spearman 0.76，对 SPURS 的 0.83、ThermoMPNN 的 0.77。我们用的是通用架构，没有任何针对稳定性的设计 |
| **任何单项上都不是最准** | AlphaMissense 在结合上赢我们的功能能（0.382 对 0.310），在临床致病性上也赢（ClinVar AUROC 0.963 对 0.921） |
| **不是临床工具** | 14e 的标签是细胞实验结果，拿来当机器学习基准用。这里没有任何东西能作为对一个人的判断依据 |
| **受限于成对数据** | 两头都测过的蛋白只有 9 个，这是拆分能力的瓶颈，也是"你会怎么改进"的诚实答案 |
| **位点地图只覆盖 118 个蛋白** | 不是整个蛋白组。扩展它是改一个列表 + 大约 6 小时算力，不是新工作 |
| **构象开关在模型之外** | SHP2 的变构隧道找不到（p = 0.39），有物理原因 |

> 一条可以背下来的原则：**把输了的数字和赢了的数字放在同一句话里说。** 这件事会让人相信你赢的那些。

---

## 10 · 压成五句话：药企面试官最想听哪一个，为什么

如果只剩 3 分钟，按这个顺序讲。**顺序不是按重要性，是按"能证明什么"。**

### 第 1 句 · 识别性问题（最值钱）

> "最直接的两头输出做出来是失败的——两个能量相关 0.82，等于没分开。我想清楚了原因：活性实验只看得见两个能量的乘积，所以这个拆分在数学上是不可识别的。我试了两个标准解法：L1 惩罚会让功能能直接塌成 0，MoCHI 式的钉住折叠能只把 0.82 挪到 0.79。真正的约束是——只有**两种方式都测过**的蛋白才能教会模型分离，训练集里有 9 个。把读出层只在这 9 个上拟合，拆分就成立了，代价是单项精度从 0.430 掉到 0.368。"

**为什么排第一**：这是整份材料里**唯一**能证明"这人能在没有答案的情况下自己走完一轮研究"的段落。实现一个已知想法谁都能做；发现一个隐藏的失败模式、用数字否掉两个标准解法、再找到真正的约束——这是在招人的时候真正稀缺的东西。

---

### 第 2 句 · KRAS 上那个反例

> "在 KRAS 上——它和唯一的近亲 HRAS 都在锁死的考试集里——ESM-2 对折叠的相关是 0.366，对结合是 0.362，**同一个答案**。我们的两个能量相差 0.23，而且方向相反。7 个实测自由能，每一个都是对应的那个能量赢，按位置重抽 6 个显著。顺带说，AlphaMissense 在结合这一项上比我们准。"

**为什么排第二**：它把一个听起来很虚的卖点（"我们输出两个数"）变成了**一个可测量的对照实验**。而且它展示了"会设计一个能证伪自己的检验"。最后那句主动认输，是信任的来源。

---

### 第 3 句 · 能量可加

> "自由能是可加的，这是热力学给的。所以只用单突变定出每个替换的能量，就能预测大约 5,000 个从没拟合过的双突变，R² 0.74 和 0.84。换成把适应度相加，R² 是**负的**——因为表型会饱和。"

**为什么排第三**：这是**最直接能落到项目上**的一条。任何蛋白工程项目都在叠突变，这句话等于说"你们现在做的那件事，我这个模型能帮上"。负 R² 那个对照让它变得很有画面感。

---

### 第 4 句 · 真药上的验证，以及被数据纠正的方向

> "migalastat 是获批的法布里病药，它的厂家公布了 GLA 变体里哪些有效哪些无效。GLA 不在我们任何切分里。两个能量做到 AUROC 0.769，零样本语言模型是 0.725。方向和我原来猜的相反——救得回的是'轻度失稳、活性位点完好'，不是'折叠全坏、功能完好'，因为这个药结合在活性位点上。"

**为什么排第四**：它把"机制"落到了一个**真实的药物决策**上，而不是停在基准数据集。后半句的自我纠正，是整段话里最可信的部分。

---

### 第 5 句 · 验证的严谨

> "蛋白留出是整个蛋白连同亲戚一起留，切分写进 dbt seed，考卷只开过一次。每一章都要拿自己带来的新数据重新查一遍——14a 查出 56 个结构域和考试蛋白同源，所以我又训了一个去污染的模型给后面所有章用，代价是 0.010 Spearman，写在文档里。"

**为什么排第五**：它不证明你聪明，它证明**你的数字可以信**。所以它不该放在最前面（那样像在辩解），但一定要在前五句里出现——否则前面四句的数字都是悬空的。

---

### 最不该讲的三件事

| 别讲 | 为什么 |
|---|---|
| 网络结构、层数、参数量 | 我们用的是通用架构，这正是我们**没有**创新的地方。讲了会把话题引到你最弱的地方 |
| "我们达到了 SOTA" | 我们没有，而且对方查得到。一旦被抓到，前面所有数字都打折 |
| 平台用了多少种 AWS 服务 | 服务清单不等于能力。讲"CI 被限流，我做了结构性修复并验证镜像摘要一致"这种具体的 |

---

## 11 · 最容易被问倒的三个问题，和怎么答

### Q1 · "这和 DETANGO 有什么区别？它也是用语言模型拆稳定性和功能。"

**先承认同题，再说三个具体差别**（都来自它自己的论文，不是我们编的）：

| | DETANGO | 我们 |
|---|---|---|
| 训练方式 | **一个蛋白一个模型**，在蛋白内部做交叉验证 | 跨蛋白训练，能预测完全没数据的蛋白 |
| 输出标度 | 用学出来的网络替掉了玻尔兹曼关系，输出是对数几率 | **kcal/mol**，能和量热法、和 ddPCA 推的能量、和 ThermoMPNN 直接比 |
| 监督信号 | ESM-1v 的对数几率 + FoldX 的**预测值** | 实验测量值，分离由**测定类型**强制 |

而且它作者自己写的是：这个分离"是操作层面的，不是严格保证的"。

> **别踩的坑**：不要说"我们比它好"。说"我们做的是另一个方向：它是每个蛋白精拟合，我们是对没数据的蛋白做预测"。

---

### Q2 · "9 个蛋白能说明什么？"

**这是最危险的问题，因为数字确实很小。** 答法是**把它从统计问题改成结构问题**：

> "9 不是用来做统计功效的，它是用来**解锁识别性**的。只有活性数据的时候，拆分有无穷多个等价解，再多数据也不会收敛到对的那个——这是不可识别，不是样本不足。有了同时测过两种方式的蛋白，解才唯一。9 个足以把读出层定下来，但它确实是这个模型现在最硬的约束，所以**我自己搭了一条从原始 FASTQ 到适应度的流水线**（14b，复现一篇发表的实验 Pearson 1.0000），就是为了能多产出这种成对数据。"

这个答法的好处：承认了弱点，解释了为什么弱点不是致命的，并且证明你**已经在动手解决它了**。

---

### Q3 · "你的绝对精度不如 SPURS，那我为什么要用你的？"

**不要辩护精度。换问题。**

> "如果你的问题是'这个突变有多不稳定'，那你就该用 SPURS，我的文档里就是这么写的。我们的输出回答的是另一个问题：**坏的是哪一部分**。举个具体的：一个变体丰度低，这是该上伴侣分子还是该放弃这条线？SPURS 和 AlphaMissense 的输出在数学上答不了这个问题，因为一个数只有一个值。我们在 migalastat 公布的名单上验过这件事。"

然后立刻给一个**什么时候该用别人的**的表——这会让你听起来像个会做技术选型的人，而不是在推销自己的模型。

---

## 小结

| 创新点 | 一句话 | 证据在哪 |
|---|---|---|
| 两个能量，不是一个分数 | 分离由测定类型强制，输出是 kcal/mol | [14c](14c_two_energies.ipynb) |
| 找到并解决了识别性问题 | 只有 9 个两头测过的蛋白能教会模型分离；两个标准解法都用数字否掉了 | [14c](14c_two_energies.ipynb) |
| 能量可加 | 单突变预测 5,000 个双突变，R² 0.74 / 0.84；适应度相加是负的 | [14c](14c_two_energies.ipynb) |
| 位点地图 | 功能位置富集 2.84 倍、结构位置耗竭到 0.21 倍；两个已成药口袋前瞻性点亮 | [14d](14d_site_map.ipynb) |
| 真药验证 | migalastat 名单 AUROC 0.769 对 0.725；方向被数据纠正 | [14e](14e_chaperone_rescue.ipynb) |
| 测多少才值 | 约 96 个是分水岭；主动学习多找 75% 的顶尖变体 | [14x](14x_low_n_and_design.ipynb) |
| kcal/mol 的精度 | 每结构域中位 Spearman 0.76、RMSE 0.74 kcal/mol | [14a](14a_stability_kcal.ipynb) |
| 上游数据自己处理 | 从原始 FASTQ 复现一篇发表的实验，Pearson 1.0000 | [14b](14b_reads_to_fitness.ipynb) |
| 验证经得起审计 | 整蛋白 + 亲戚留出、考卷只开一次、每章自查污染并报代价 | 07a、14a |
| 真的上线了 | 2.18 亿变体进湖、serverless API、带护栏的 agent、88 个 Terraform 资源、263 个测试 | 09c、10、13 |

## 去哪看别的

| 你想要 | 看这里 |
|---|---|
| 同样的内容，英文、给别人看的版本 | [`docs/contributions.md`](../../docs/contributions.md) |
| 所有数字，一页 | [`docs/phase_f_results.md`](../../docs/phase_f_results.md) |
| 为什么不直接用 AlphaMissense / SPURS | [`docs/why_not_just_use_alphamissense.md`](../../docs/why_not_just_use_alphamissense.md) |
| 按岗位的 30 秒 / 2 分钟讲稿和简历条目 | [`docs/interview_pack.md`](../../docs/interview_pack.md) |